# Part 13: ML Ethics & Explainability

[← Back to Index](Index.ipynb)

**Quick Reference Guide for Responsible AI and Model Interpretability**

---
## 13.1 Bias & Fairness

**Types of Bias:**

1. **Historical Bias:** Bias in training data reflects past discrimination
2. **Representation Bias:** Underrepresentation of certain groups
3. **Measurement Bias:** Features measured differently across groups
4. **Aggregation Bias:** One-size-fits-all model for diverse groups
5. **Evaluation Bias:** Benchmark doesn't represent use case
6. **Deployment Bias:** Model used in different context than trained for

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Generate synthetic dataset with bias
np.random.seed(42)
n_samples = 1000

# Create biased data: Group A has higher approval rate than Group B
df = pd.DataFrame({
    'credit_score': np.random.randint(300, 850, n_samples),
    'income': np.random.randint(20000, 150000, n_samples),
    'age': np.random.randint(18, 70, n_samples),
    'protected_group': np.random.choice([0, 1], n_samples)  # 0=Group A, 1=Group B
})

# Introduce bias: approval depends on protected group
df['approved'] = (
    (df['credit_score'] > 600) & 
    (df['income'] > 40000) &
    ((df['protected_group'] == 0) | (np.random.random(n_samples) > 0.3))  # Bias
).astype(int)

print("Dataset with bias:")
print(df.head())
print(f"\nApproval rate by group:")
print(df.groupby('protected_group')['approved'].mean())

---
## 13.2 Fairness Metrics

**Key Metrics:**

In [ ]:
class FairnessMetrics:
    """Calculate fairness metrics"""
    
    @staticmethod
    def demographic_parity(y_pred, sensitive_feature):
        """
        Demographic Parity (Statistical Parity)
        P(Y_hat=1 | A=0) = P(Y_hat=1 | A=1)
        """
        group0_rate = y_pred[sensitive_feature == 0].mean()
        group1_rate = y_pred[sensitive_feature == 1].mean()
        
        ratio = min(group0_rate, group1_rate) / max(group0_rate, group1_rate)
        difference = abs(group0_rate - group1_rate)
        
        return {
            'group0_positive_rate': group0_rate,
            'group1_positive_rate': group1_rate,
            'ratio': ratio,
            'difference': difference
        }
    
    @staticmethod
    def equal_opportunity(y_true, y_pred, sensitive_feature):
        """
        Equal Opportunity
        TPR should be equal across groups
        P(Y_hat=1 | Y=1, A=0) = P(Y_hat=1 | Y=1, A=1)
        """
        # True Positive Rate for each group
        tpr0 = ((y_pred == 1) & (y_true == 1) & (sensitive_feature == 0)).sum() / \
               ((y_true == 1) & (sensitive_feature == 0)).sum()
        
        tpr1 = ((y_pred == 1) & (y_true == 1) & (sensitive_feature == 1)).sum() / \
               ((y_true == 1) & (sensitive_feature == 1)).sum()
        
        ratio = min(tpr0, tpr1) / max(tpr0, tpr1) if max(tpr0, tpr1) > 0 else 0
        difference = abs(tpr0 - tpr1)
        
        return {
            'group0_tpr': tpr0,
            'group1_tpr': tpr1,
            'ratio': ratio,
            'difference': difference
        }
    
    @staticmethod
    def equalized_odds(y_true, y_pred, sensitive_feature):
        """
        Equalized Odds
        Both TPR and FPR should be equal across groups
        """
        # TPR
        tpr0 = ((y_pred == 1) & (y_true == 1) & (sensitive_feature == 0)).sum() / \
               max(((y_true == 1) & (sensitive_feature == 0)).sum(), 1)
        tpr1 = ((y_pred == 1) & (y_true == 1) & (sensitive_feature == 1)).sum() / \
               max(((y_true == 1) & (sensitive_feature == 1)).sum(), 1)
        
        # FPR
        fpr0 = ((y_pred == 1) & (y_true == 0) & (sensitive_feature == 0)).sum() / \
               max(((y_true == 0) & (sensitive_feature == 0)).sum(), 1)
        fpr1 = ((y_pred == 1) & (y_true == 0) & (sensitive_feature == 1)).sum() / \
               max(((y_true == 0) & (sensitive_feature == 1)).sum(), 1)
        
        return {
            'group0_tpr': tpr0,
            'group1_tpr': tpr1,
            'tpr_difference': abs(tpr0 - tpr1),
            'group0_fpr': fpr0,
            'group1_fpr': fpr1,
            'fpr_difference': abs(fpr0 - fpr1)
        }
    
    @staticmethod
    def disparate_impact(y_pred, sensitive_feature, threshold=0.8):
        """
        Disparate Impact (80% rule)
        Ratio should be >= 0.8
        """
        dp = FairnessMetrics.demographic_parity(y_pred, sensitive_feature)
        passes_80_rule = dp['ratio'] >= threshold
        
        return {
            **dp,
            'passes_80_rule': passes_80_rule
        }

# Train a model
X = df[['credit_score', 'income', 'age', 'protected_group']]
y = df['approved']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = LogisticRegression(random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Calculate fairness metrics
sensitive = X_test['protected_group'].values

print("\n=== Fairness Metrics ===")
print("\n1. Demographic Parity:")
dp = FairnessMetrics.demographic_parity(y_pred, sensitive)
for k, v in dp.items():
    print(f"   {k}: {v:.3f}")

print("\n2. Equal Opportunity:")
eo = FairnessMetrics.equal_opportunity(y_test.values, y_pred, sensitive)
for k, v in eo.items():
    print(f"   {k}: {v:.3f}")

print("\n3. Equalized Odds:")
eq = FairnessMetrics.equalized_odds(y_test.values, y_pred, sensitive)
for k, v in eq.items():
    print(f"   {k}: {v:.3f}")

print("\n4. Disparate Impact:")
di = FairnessMetrics.disparate_impact(y_pred, sensitive)
print(f"   Ratio: {di['ratio']:.3f}")
print(f"   Passes 80% rule: {di['passes_80_rule']}")

### Visualize Fairness

In [ ]:
# Visualize bias
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrices by group
for group in [0, 1]:
    mask = sensitive == group
    cm = confusion_matrix(y_test[mask], y_pred[mask])
    
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[group])
    axes[group].set_title(f'Group {group} Confusion Matrix')
    axes[group].set_xlabel('Predicted')
    axes[group].set_ylabel('Actual')

plt.tight_layout()
plt.show()

# Approval rates
results = pd.DataFrame({
    'Group': ['Group 0', 'Group 1'],
    'Approval Rate': [dp['group0_positive_rate'], dp['group1_positive_rate']]
})

plt.figure(figsize=(8, 5))
plt.bar(results['Group'], results['Approval Rate'], color=['blue', 'orange'])
plt.axhline(y=0.8*results['Approval Rate'].max(), color='red', 
            linestyle='--', label='80% threshold')
plt.ylabel('Approval Rate')
plt.title('Approval Rate by Protected Group')
plt.legend()
plt.ylim(0, 1)
plt.grid(axis='y', alpha=0.3)
plt.show()

---
## 13.3 Model Explainability

**Why Explainability?**
- Build trust
- Debug models
- Regulatory compliance (GDPR, etc.)
- Detect bias
- Improve models

**Types:**
1. **Global:** Overall model behavior
2. **Local:** Individual prediction explanation

### Permutation Importance

**Concept:** Measure feature importance by shuffling feature values

In [ ]:
from sklearn.inspection import permutation_importance

# Calculate permutation importance
perm_importance = permutation_importance(
    model, X_test, y_test,
    n_repeats=10,
    random_state=42
)

# Create DataFrame
importance_df = pd.DataFrame({
    'feature': X.columns,
    'importance': perm_importance.importances_mean,
    'std': perm_importance.importances_std
}).sort_values('importance', ascending=False)

print("Permutation Importance:")
print(importance_df)

# Visualize
plt.figure(figsize=(10, 6))
plt.barh(importance_df['feature'], importance_df['importance'], 
         xerr=importance_df['std'])
plt.xlabel('Importance')
plt.title('Permutation Feature Importance')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

### SHAP (SHapley Additive exPlanations)

**Concept:** Game theory approach to explain predictions

**Advantages:**
- Theoretically sound
- Consistent
- Local and global explanations

In [ ]:
import shap

# Create explainer
explainer = shap.LinearExplainer(model, X_train)
shap_values = explainer.shap_values(X_test)

# Summary plot (global)
print("SHAP Summary Plot (shows feature importance and effects):")
shap.summary_plot(shap_values, X_test, plot_type="bar", show=False)
plt.tight_layout()
plt.show()

# Detailed summary
shap.summary_plot(shap_values, X_test, show=False)
plt.tight_layout()
plt.show()

# Individual prediction explanation
print("\nExplanation for first test instance:")
shap.force_plot(
    explainer.expected_value, 
    shap_values[0], 
    X_test.iloc[0],
    matplotlib=True,
    show=False
)
plt.tight_layout()
plt.show()

### LIME (Local Interpretable Model-agnostic Explanations)

**Concept:** Explain individual predictions by approximating model locally with interpretable model

In [ ]:
from lime import lime_tabular

# Create LIME explainer
lime_explainer = lime_tabular.LimeTabularExplainer(
    X_train.values,
    feature_names=X_train.columns.tolist(),
    class_names=['Rejected', 'Approved'],
    mode='classification'
)

# Explain a prediction
idx = 0
exp = lime_explainer.explain_instance(
    X_test.iloc[idx].values,
    model.predict_proba,
    num_features=4
)

print(f"\nLIME Explanation for instance {idx}:")
print(f"Prediction: {y_pred[idx]} (Actual: {y_test.iloc[idx]})")
print(f"\nFeature contributions:")
for feature, weight in exp.as_list():
    print(f"  {feature}: {weight:.3f}")

# Visualize
exp.as_pyplot_figure()
plt.tight_layout()
plt.show()

### Partial Dependence Plots (PDP)

**Concept:** Show marginal effect of features on predictions

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

# Partial Dependence Plot
features = ['credit_score', 'income', 'age']
fig, ax = plt.subplots(figsize=(14, 4))

display = PartialDependenceDisplay.from_estimator(
    model, X_test, features,
    ax=ax
)

plt.suptitle('Partial Dependence Plots')
plt.tight_layout()
plt.show()

---
## 13.4 Privacy-Preserving ML

**Techniques:**

### Differential Privacy

**Concept:** Add noise to protect individual privacy while maintaining statistical properties

In [ ]:
class DifferentialPrivacy:
    """Simple differential privacy implementation"""
    
    @staticmethod
    def laplace_mechanism(value, sensitivity, epsilon):
        """
        Add Laplace noise for differential privacy
        
        epsilon: privacy budget (smaller = more private)
        sensitivity: max change in output from single record
        """
        scale = sensitivity / epsilon
        noise = np.random.laplace(0, scale)
        return value + noise
    
    @staticmethod
    def private_mean(data, epsilon=1.0):
        """Calculate mean with differential privacy"""
        true_mean = np.mean(data)
        
        # Assuming data is bounded [0, 1]
        sensitivity = 1.0 / len(data)
        
        private_mean = DifferentialPrivacy.laplace_mechanism(
            true_mean, sensitivity, epsilon
        )
        
        return private_mean, true_mean
    
    @staticmethod
    def private_count(data, threshold, epsilon=1.0):
        """Count with differential privacy"""
        true_count = np.sum(data > threshold)
        
        # Sensitivity is 1 (one person can change count by 1)
        sensitivity = 1.0
        
        private_count = DifferentialPrivacy.laplace_mechanism(
            true_count, sensitivity, epsilon
        )
        
        return max(0, private_count), true_count

# Example
data = np.random.random(1000)

# Private mean
print("Differential Privacy Examples:\n")
for epsilon in [0.1, 1.0, 10.0]:
    private, true = DifferentialPrivacy.private_mean(data, epsilon)
    error = abs(private - true)
    print(f"Epsilon={epsilon:4.1f}: Private={private:.4f}, True={true:.4f}, Error={error:.4f}")

# Private count
print("\nPrivate count (threshold=0.5):")
for epsilon in [0.1, 1.0, 10.0]:
    private, true = DifferentialPrivacy.private_count(data, 0.5, epsilon)
    print(f"Epsilon={epsilon:4.1f}: Private={private:.1f}, True={true}")

### Federated Learning

**Concept:** Train model across decentralized data without sharing raw data

**Benefits:**
- Privacy preservation
- Reduced data transfer
- Compliance with regulations

In [ ]:
class FederatedLearning:
    """Simple federated learning simulation"""
    
    def __init__(self, n_clients):
        self.n_clients = n_clients
        self.global_model = None
        self.client_models = []
    
    def distribute_data(self, X, y):
        """Distribute data across clients"""
        client_data = []
        
        # Split data randomly
        indices = np.random.permutation(len(X))
        splits = np.array_split(indices, self.n_clients)
        
        for split in splits:
            client_data.append((X.iloc[split], y.iloc[split]))
        
        return client_data
    
    def train_client(self, X_client, y_client):
        """Train model on client data"""
        model = LogisticRegression(random_state=42, max_iter=100)
        model.fit(X_client, y_client)
        return model
    
    def aggregate_models(self, client_models):
        """Federated averaging"""
        # Average model coefficients
        avg_coef = np.mean([m.coef_ for m in client_models], axis=0)
        avg_intercept = np.mean([m.intercept_ for m in client_models], axis=0)
        
        # Create global model
        global_model = LogisticRegression()
        global_model.coef_ = avg_coef
        global_model.intercept_ = avg_intercept
        global_model.classes_ = client_models[0].classes_
        
        return global_model
    
    def train_federated(self, X, y, rounds=3):
        """Federated training"""
        print(f"Training federated model with {self.n_clients} clients\n")
        
        client_data = self.distribute_data(X, y)
        
        for round_num in range(rounds):
            print(f"Round {round_num + 1}/{rounds}")
            
            # Train on each client
            client_models = []
            for i, (X_client, y_client) in enumerate(client_data):
                model = self.train_client(X_client, y_client)
                client_models.append(model)
                print(f"  Client {i+1} trained on {len(X_client)} samples")
            
            # Aggregate
            self.global_model = self.aggregate_models(client_models)
            print(f"  Models aggregated\n")
        
        return self.global_model

# Example
fed_learning = FederatedLearning(n_clients=3)
federated_model = fed_learning.train_federated(X_train, y_train, rounds=2)

# Evaluate
y_pred_fed = federated_model.predict(X_test)
accuracy_fed = accuracy_score(y_test, y_pred_fed)
print(f"Federated model accuracy: {accuracy_fed:.3f}")

# Compare with centralized
print(f"Centralized model accuracy: {accuracy_score(y_test, y_pred):.3f}")

---
## 13.5 Responsible AI Checklist

**Before Deployment:**

1. **Fairness:**
   - [ ] Identify protected attributes
   - [ ] Measure fairness metrics
   - [ ] Test across different groups
   - [ ] Document bias mitigation strategies

2. **Explainability:**
   - [ ] Can you explain individual predictions?
   - [ ] Do you understand feature importance?
   - [ ] Can stakeholders understand the model?
   - [ ] Document model limitations

3. **Privacy:**
   - [ ] Data minimization
   - [ ] Anonymization/pseudonymization
   - [ ] Consider differential privacy
   - [ ] Secure data storage and transmission

4. **Robustness:**
   - [ ] Test on adversarial examples
   - [ ] Handle edge cases
   - [ ] Monitor for drift
   - [ ] Uncertainty quantification

5. **Accountability:**
   - [ ] Clear ownership
   - [ ] Audit trail
   - [ ] Human oversight
   - [ ] Feedback mechanism

---
### Quick Reference Guide

**Fairness Definitions:**

| Metric | Definition | When to Use |
|--------|-----------|-------------|
| **Demographic Parity** | Equal positive rate across groups | Equal opportunity context |
| **Equal Opportunity** | Equal TPR across groups | When FN cost is high |
| **Equalized Odds** | Equal TPR and FPR across groups | Balanced fairness |
| **Predictive Parity** | Equal PPV across groups | When FP cost is high |
| **Calibration** | Predicted probabilities match actual | Risk assessment |

**Explainability Methods:**

| Method | Type | Scope | Model-Agnostic | Best For |
|--------|------|-------|----------------|----------|
| **SHAP** | Additive | Local/Global | Yes | Any model |
| **LIME** | Surrogate | Local | Yes | Black-box models |
| **PDP** | Marginal effect | Global | Yes | Feature effects |
| **Permutation** | Importance | Global | Yes | Feature ranking |
| **ICE** | Individual curves | Local | Yes | Heterogeneity |
| **Attention** | Weights | Local | No | Neural networks |

**Privacy Techniques:**

| Technique | Description | Trade-off |
|-----------|-------------|----------|
| **Differential Privacy** | Add noise to outputs | Accuracy vs privacy |
| **Federated Learning** | Decentralized training | Communication overhead |
| **Homomorphic Encryption** | Compute on encrypted data | Performance cost |
| **Secure Multi-party Computation** | Collaborative computation | Complexity |
| **k-Anonymity** | Group records | Information loss |

**Best Practices:**

1. **Document everything:** Model cards, data sheets
2. **Diverse teams:** Multiple perspectives
3. **Stakeholder involvement:** Include affected communities
4. **Regular audits:** Continuous monitoring
5. **Transparency:** Clear communication
6. **Human in the loop:** Critical decisions
7. **Contestability:** Appeal mechanism
8. **Impact assessment:** Before deployment

**Regulations:**
- GDPR (EU): Right to explanation
- CCPA (California): Data privacy
- Fair Credit Reporting Act (US): Credit decisions
- Equal Credit Opportunity Act (US): No discrimination
- EU AI Act: Risk-based regulation

---
[← Back to Index](Index.ipynb)